# SteerMoE: forcing number formatting on Qwen3-30B-A3B

Replicates the expert-deactivation mechanism of SteerMoE
([arXiv:2509.09660](https://arxiv.org/abs/2509.09660)) on
**Qwen3-30B-A3B — a model evaluated in the paper**.
`expert_detection.ipynb` identifies experts whose top-k routing is
linked to written number words vs digit formatting; deactivating the
200 word-linked experts (committed in `steermoe_qwen3_words.json`)
flips greedy counting from words to **digits**: the baseline answers
"One, two, three…", the steered model "1, 2, 3…".

The effect saturates (digit-share 1.00 on both prompts below), so it
is robust to run-to-run and GPU numerics. The mechanism check at the
end verifies zero deactivated-expert selections post-steering.

Qwen3-30B-A3B is a hybrid thinking model; prompts render with
`enable_thinking=False`, as in the official SteerMoE code.


In [1]:
import json
import os

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
# Qwen3-MoE is outside the default V2-runner list; the mechanism
# check captures router logits, which requires the V2 runner.
os.environ.setdefault("VLLM_USE_V2_MODEL_RUNNER", "1")

import numpy as np

from vllm import LLM, SamplingParams
from vllm.capture import deserialize_captured
from vllm.steer_vectors import ApplySpec, SteeringSpec, VectorSpec

MODEL = os.path.expanduser("/data/zju-130/shenyl/hf/model/Qwen/Qwen3-30B-A3B")  # Qwen/Qwen3-30B-A3B

# Eager mode for the router-logit mechanism check; prefix caching
# off because cache-hit tokens are never recomputed and so could
# never be captured.
llm = LLM(
    model=MODEL,
    enable_steer_vector=True,
    enforce_eager=True,
    enable_chunked_prefill=False,
    enable_prefix_caching=False,
    gpu_memory_utilization=0.92,
    max_model_len=2048,
    max_num_seqs=4,
)
tok = llm.get_tokenizer()


def rpc(method, *args, **kwargs):
    return llm.llm_engine.collective_rpc(method, args=args, kwargs=kwargs)[0]


/home/xhl/anaconda3/envs/easysteer-vllm026/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/home/xhl/anaconda3/envs/easysteer-vllm026/lib/python3.12/site-packages/pydantic/dataclasses.py:313: UserWarning: `config` is set via both the `dataclass` decorator and `__pydantic_config__` for dataclass SteerVectorConfig. The `config` specification from `dataclass` decorator will take priority.
  return create_dataclass if _cls is None else create_dataclass(_cls)


INFO 08-04 01:49:58 [api_utils.py:273] non-default args: {'max_model_len': 2048, 'enable_prefix_caching': False, 'max_num_seqs': 4, 'disable_log_stats': True, 'enforce_eager': True, 'enable_steer_vector': True, 'enable_chunked_prefill': False, 'model': '/data/zju-130/shenyl/hf/model/Qwen/Qwen3-30B-A3B'}


INFO 08-04 01:49:59 [model.py:623] Resolved architecture: Qwen3MoeForCausalLM


INFO 08-04 01:49:59 [model.py:1788] Using max model len 2048


...

WARNING 08-04 01:51:37 [vllm.py:1249] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.


(EngineCore pid=4082142) 

INFO 08-04 01:51:37 [kernel.py:295] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c', 'native'])


(EngineCore pid=4082142) 

INFO 08-04 01:51:37 [vllm.py:1428] Cudagraph is disabled under eager mode


In [2]:
with open("steermoe_qwen3_words.json") as f:
    layer_configs = json.load(f)["layer_configs"]

steering = SteeringSpec(vectors=[
    VectorSpec(
        source="steermoe_qwen3_words.json",
        algorithm="moe_router",  # per-layer mode/expert_ids come from the JSON
        layers=sorted(int(la) for la in layer_configs),
        apply=ApplySpec(phases=["prompt", "generation"]),
    ),
])


def render(prompt):
    text = tok.apply_chat_template(
        [{"role": "user", "content": prompt}], tokenize=False,
        add_generation_prompt=True, enable_thinking=False,
    )
    return tok(text, add_special_tokens=False).input_ids


def gen(prompt, spec=None):
    out = llm.generate({"prompt_token_ids": render(prompt)},
                       sampling_params=SamplingParams(temperature=0.0,
                                                      max_tokens=64),
                       steering=spec)
    return out[0].outputs[0].text.strip().replace("\n", " ")


def digit_share(text):
    digits = sum(c.isdigit() for c in text)
    letters = sum(c.isalpha() for c in text)
    return digits / max(1, digits + letters)


In [3]:
PROMPTS = [
    "Count to fifteen.",
    "Count from one to twelve.",
]

for prompt in PROMPTS:
    base = gen(prompt)
    steered = gen(prompt, steering)
    print(f"[{prompt}]")
    print(f"  baseline (digit-share {digit_share(base):.2f}): {base}")
    print(f"  steered  (digit-share {digit_share(steered):.2f}): {steered}")


Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Rendering prompts: 100%|██████████| 1/1 [00:00<00:00, 25.11it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts: 100%|██████████| 1/1 [00:03<00:00,  3.72s/it, est. speed input: 4.31 toks/s, output: 17.23 toks/s]

Processed prompts: 100%|██████████| 1/1 [00:03<00:00,  3.72s/it, est. speed input: 4.31 toks/s, output: 17.23 toks/s]

...

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts: 100%|██████████| 1/1 [00:02<00:00,  2.57s/it, est. speed input: 7.00 toks/s, output: 15.18 toks/s]

Processed prompts: 100%|██████████| 1/1 [00:02<00:00,  2.57s/it, est. speed input: 7.00 toks/s, output: 15.18 toks/s]

Processed prompts: 100%|██████████| 1/1 [00:02<00:00,  2.57s/it, est. speed input: 7.00 toks/s, output: 15.18 toks/s]

[Count from one to twelve.]
  baseline (digit-share 0.00): One, two, three, four, five, six, seven, eight, nine, ten, eleven, twelve.
  steered  (digit-share 1.00): 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12.


## Mechanism check

Capture the post-steering router logits for one steered prompt and
verify that none of the deactivated experts is selected in any
token's top-8 — the deactivation is airtight; the digits come from
the model routing around the removed experts.

In [4]:
deact = {int(la): c["expert_ids"] for la, c in layer_configs.items()}
TOP_K = 8

rpc("start_capture", "router_logits")
llm.generate({"prompt_token_ids": render("Count to fifteen.")},
             sampling_params=SamplingParams(temperature=0.0, max_tokens=32),
             steering=steering)
logits = {lid: t.float().numpy()
          for lid, t in deserialize_captured(
              rpc("fetch_captured", "router_logits"))[0].items()}
rpc("stop_capture", "router_logits")

leaks = 0
for layer, expert_ids in deact.items():
    top = np.argsort(logits[layer], axis=-1)[:, -TOP_K:]
    leaks += int(np.isin(top, expert_ids).sum())
print(f"deactivated-expert selections post-steering: {leaks} "
      f"(0 = steering is airtight)")


Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Rendering prompts: 100%|██████████| 1/1 [00:00<00:00, 569.72it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts: 100%|██████████| 1/1 [00:02<00:00,  2.65s/it, est. speed input: 6.03 toks/s, output: 12.06 toks/s]

Processed prompts: 100%|██████████| 1/1 [00:02<00:00,  2.65s/it, est. speed input: 6.03 toks/s, output: 12.06 toks/s]

Processed prompts: 100%|██████████| 1/1 [00:02<00:00,  2.66s/it, est. speed input: 6.03 toks/s, output: 12.06 toks/s]

deactivated-expert selections post-steering: 0 (0 = steering is airtight)
